<a href="https://colab.research.google.com/github/Ruhul73/Email-sender-in-bulk/blob/main/Email_sender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Create a professional bulk email sender web application using Flask and pandas that allows users to upload an Excel file of recipients, input SMTP credentials, and send personalized emails, including a complete backend logic for mail merging and a frontend UI.

## Setup Environment and Project Structure

### Subtask:
Initialize the project workspace and install the required Python libraries for the Flask web application.


**Reasoning**:
I will initialize the project directory structure, install the necessary dependencies, and create the entry point for the Flask application as requested.



In [ ]:
import os

# 1. Create project directory and folder structure
project_name = 'bulk_email_sender'
os.makedirs(f'{project_name}/templates', exist_ok=True)
os.makedirs(f'{project_name}/uploads', exist_ok=True)

# 2. Install necessary libraries
!pip install Flask pandas openpyxl

# 3. Create main application file app.py with basic initialization
app_content = """
from flask import Flask, render_template, request, redirect, url_for
import os

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = 'uploads'

@app.route('/')
def index():
    return "Bulk Email Sender API is running."

if __name__ == '__main__':
    app.run(debug=True)
"""

with open(f'{project_name}/app.py', 'w') as f:
    f.write(app_content.strip())

print(f'Project {project_name} initialized with folders and app.py.')

Project bulk_email_sender initialized with folders and app.py.


## Create Frontend (UI)

### Subtask:
Develop an HTML interface with a professional look including a form for SMTP credentials, email content, and file upload.


**Reasoning**:
I will create the 'index.html' template with a professional design and update the Flask application to render this template.



In [ ]:
index_html_content = """
<!DOCTYPE html>
<html lang='en'>
<head>
    <meta charset='UTF-8'>
    <meta name='viewport' content='width=device-width, initial-scale=1.0'>
    <title>Bulk Email Sender</title>
    <link href='https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css' rel='stylesheet'>
    <style>
        body { background-color: #f8f9fa; padding-top: 50px; }
        .container { max-width: 800px; background: white; padding: 30px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
        .form-label { font-weight: bold; }
        .btn-primary { width: 100%; }
    </style>
</head>
<body>
<div class='container'>
    <h2 class='text-center mb-4'>Bulk Email Sender</h2>
    <form action='/send-email' method='POST' enctype='multipart/form-data'>
        <div class='row mb-3'>
            <div class='col-md-6'>
                <label class='form-label'>SMTP Server</label>
                <input type='text' name='smtp_server' class='form-control' placeholder='smtp.gmail.com' required>
            </div>
            <div class='col-md-6'>
                <label class='form-label'>Port</label>
                <input type='number' name='smtp_port' class='form-control' placeholder='587' required>
            </div>
        </div>
        <div class='row mb-3'>
            <div class='col-md-6'>
                <label class='form-label'>Sender Email</label>
                <input type='email' name='email_user' class='form-control' required>
            </div>
            <div class='col-md-6'>
                <label class='form-label'>Password</label>
                <input type='password' name='email_pass' class='form-control' required>
            </div>
        </div>
        <hr>
        <div class='mb-3'>
            <label class='form-label'>Recipient List (Excel or CSV)</label>
            <input type='file' name='file' class='form-control' accept='.xlsx, .xls, .csv' required>
            <small class='text-muted'>File must contain an 'Email' column. Use `{{name}}` in body for personalization.</small>
        </div>
        <div class='mb-3'>
            <label class='form-label'>Email Subject</label>
            <input type='text' name='subject' class='form-control' required>
        </div>
        <div class='mb-3'>
            <label class='form-label'>Email Body</label>
            <textarea name='body' class='form-control' rows='5' placeholder='Hello {{name}}, ...' required></textarea>
        </div>
        <div class='mb-3'>
            <label class='form-label'>Attachment (Optional)</label>
            <input type='file' name='attachment' class='form-control'>
            <small class='text-muted'>Select a file to attach to all emails.</small>
        </div>
        <button type='submit' class='btn btn-primary'>Start Sending Emails</button>
    </form>
</div>
</body>
</html>
"""

# Write the index.html file
with open(f'{project_name}/templates/index.html', 'w') as f:
    f.write(index_html_content.strip())

# Create empty configure_smtp.html and compose_email.html (or remove them) to avoid confusion
with open(f'{project_name}/templates/configure_smtp.html', 'w') as f:
    f.write('')
with open(f'{project_name}/templates/compose_email.html', 'w') as f:
    f.write('')

# Update app.py to render the NEW template configure_smtp.html
updated_app_content = """
from flask import Flask, render_template, request, redirect, url_for
import os

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = 'uploads'

@app.route('/')
def index():
    return render_template('index.html')

if __name__ == '__main__':
    app.run(debug=True)
"""

with open(f'{project_name}/app.py', 'w') as f:
    f.write(updated_app_content.strip())

print('Frontend UI simplified to a single index.html. Flask app.py updated to render index.html.')

Frontend UI simplified to a single index.html. Flask app.py updated to render index.html.


## Backend Excel Processing

### Subtask:
Implement logic to handle file uploads and extract recipient data from Excel files using pandas.


**Reasoning**:
I will update the 'app.py' file to include the backend logic for handling file uploads, processing SMTP form data, and extracting recipient details from the Excel file using pandas as specified in the instructions.



In [ ]:
import pandas as pd
from werkzeug.utils import secure_filename

# Define the updated app.py content with backend processing logic
backend_logic_content = """
from flask import Flask, render_template, request, redirect, url_for
import os
import pandas as pd
from werkzeug.utils import secure_filename

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = 'uploads'
os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/send-email', methods=['POST'])
def send_email():
    # 1. Capture SMTP Credentials and Email Content
    smtp_server = request.form.get('smtp_server')
    smtp_port = request.form.get('smtp_port')
    email_user = request.form.get('email_user')
    email_pass = request.form.get('email_pass')
    subject = request.form.get('subject')
    body_template = request.form.get('body')

    # 2. Handle File Upload
    if 'file' not in request.files:
        return "No file part"
    file = request.files['file']
    if file.filename == '':
        return "No selected file"

    if file:
        filename = secure_filename(file.filename)
        filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(filepath)

        # 3. Process Excel file using pandas
        try:
            df = pd.read_excel(filepath)

            if 'Email' not in df.columns:
                return "Error: Excel file must contain an 'Email' column."

            recipients = []
            for _, row in df.iterrows():
                recipient_info = {
                    'email': str(row['Email']).strip(),
                    'name': str(row.get('Name', 'Customer')).strip() if pd.notnull(row.get('Name')) else 'Customer'
                }
                recipients.append(recipient_info)

            # For verification in this subtask
            print(f'Processed {len(recipients)} recipients.')
            return f"Successfully parsed {len(recipients)} recipients. Ready to send emails."

        except Exception as e:
            return f"An error occurred while processing the file: {str(e)}"

if __name__ == '__main__':
    app.run(debug=True)
"""

# Write the updated logic to app.py
with open(f'{project_name}/app.py', 'w') as f:
    f.write(backend_logic_content.strip())

print('Backend processing logic implemented in app.py.')

Backend processing logic implemented in app.py.


## Email Sending Logic

### Subtask:
Implement a secure SMTP connection to send personalized bulk emails with error handling.


**Reasoning**:
I will update 'app.py' to include the full email sending logic, incorporating smtplib for secure SMTP connections, mail merging, and error handling as specified in the instructions.



In [ ]:
import smtplib
from email.message import EmailMessage
import mimetypes # Import mimetypes for handling file types

# Define the final complete app.py content
final_app_content = """
from flask import Flask, render_template, request, redirect, url_for
import os
import pandas as pd
from werkzeug.utils import secure_filename
import smtplib
from email.message import EmailMessage
import mimetypes

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = 'uploads'
os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)

def send_individual_email(smtp_server, smtp_port, user, password, recipient_email, subject, body, attachment_path=None):
    msg = EmailMessage()
    msg.set_content(body)
    msg['Subject'] = subject
    msg['From'] = user
    msg['To'] = recipient_email

    if attachment_path:
        # Guess the content type of the attachment
        ctype, encoding = mimetypes.guess_type(attachment_path)
        if ctype is None or encoding is not None:
            ctype = 'application/octet-stream'

        maintype, subtype = ctype.split('/', 1)

        with open(attachment_path, 'rb') as fp:
            msg.add_attachment(fp.read(),
                               maintype=maintype,
                               subtype=subtype,
                               filename=os.path.basename(attachment_path))

    with smtplib.SMTP(smtp_server, int(smtp_port)) as server:
        server.starttls()
        server.login(user, password)
        server.send_message(msg)

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/send-email', methods=['POST'])
def send_email():
    smtp_server = request.form.get('smtp_server')
    smtp_port = request.form.get('smtp_port')
    email_user = request.form.get('email_user')
    email_pass = request.form.get('email_pass')
    subject = request.form.get('subject')
    body_template = request.form.get('body')

    if not all([smtp_server, smtp_port, email_user, email_pass, subject, body_template]):
        return "Please provide all required fields (SMTP, subject, body).", 400

    # Handle recipient list file upload
    if 'file' not in request.files:
        return "No recipient file part.", 400
    recipient_file = request.files['file']
    if recipient_file.filename == '':
        return "No selected recipient file.", 400

    recipient_filepath = None
    attachment_filepath = None

    try:
        # Save recipient file
        recipient_filename = secure_filename(recipient_file.filename)
        recipient_filepath = os.path.join(app.config['UPLOAD_FOLDER'], recipient_filename)
        recipient_file.save(recipient_filepath)

        # Handle optional attachment file upload
        attachment_file = request.files.get('attachment')
        if attachment_file and attachment_file.filename != '':
            attachment_filename = secure_filename(attachment_file.filename)
            attachment_filepath = os.path.join(app.config['UPLOAD_FOLDER'], attachment_filename)
            attachment_file.save(attachment_filepath)

        # Determine file type and read accordingly
        df = None
        if recipient_filename.endswith(('.xlsx', '.xls')):
            df = pd.read_excel(recipient_filepath)
        elif recipient_filename.endswith('.csv'):
            df = pd.read_csv(recipient_filepath)
        else:
            return "Error: Unsupported recipient file type. Please upload an Excel (.xlsx, .xls) or CSV (.csv) file.", 400

        if 'Email' not in df.columns:
            return "Error: Recipient file must contain an 'Email' column.", 400

        success_count = 0
        failure_count = 0

        for _, row in df.iterrows():
            recipient_email = str(row['Email']).strip()
            recipient_name = str(row.get('Name', 'Customer')).strip() if pd.notnull(row.get('Name')) else 'Customer'

            # Mail Merge
            personalized_body = body_template.replace('{{name}}', recipient_name)

            try:
                send_individual_email(
                    smtp_server, smtp_port, email_user, email_pass,
                    recipient_email, subject, personalized_body, attachment_filepath
                )
                success_count += 1
            except Exception as e:
                print(f"Failed to send to {recipient_email}: {e}")
                failure_count += 1

        return f"Bulk sending complete. Sent: {success_count}, Failed: {failure_count}."

    except Exception as e:
        return f"An error occurred: {str(e)}", 500
    finally:
        # Clean up uploaded files
        if recipient_filepath and os.path.exists(recipient_filepath):
            os.remove(recipient_filepath)
        if attachment_filepath and os.path.exists(attachment_filepath):
            os.remove(attachment_filepath)

if __name__ == '__main__':
    app.run(debug=True)
"""

# Overwrite app.py with the full implementation
with open(f'{project_name}/app.py', 'w') as f:
    f.write(final_app_content.strip())

print('Email sending logic updated in app.py for simplified single-page flow.')

Email sending logic updated in app.py for simplified single-page flow.


## Testing and Link Generation Guide

### Subtask:
Create a script to run the Flask application in the Google Colab environment using a local tunnel and provide a testing guide.


**Reasoning**:
I will create a script to run the Flask application in the background using threading, expose it via a public URL using google.colab.output, and generate a sample Excel file for testing.



## Final Task

### Subtask:
Summarize the tool features and provide the final professional code package for your client.


In [ ]:
import threading
import pandas as pd
from google.colab import output
from bulk_email_sender.app import app

# 1. Create a sample Excel and CSV file for testing
sample_data = {
    'Name': ['John Doe', 'Jane Smith', 'Alice Wonderland'],
    'Email': ['john.doe@example.com', 'jane.smith@example.com', 'alice@example.com']
}
df_sample = pd.DataFrame(sample_data)

# Create sample Excel file
sample_excel_file_path = 'sample_recipients.xlsx'
df_sample.to_excel(sample_excel_file_path, index=False)
print(f'Sample testing Excel file created at: {sample_excel_file_path}')

# Create sample CSV file
sample_csv_file_path = 'sample_recipients.csv'
df_sample.to_csv(sample_csv_file_path, index=False)
print(f'Sample testing CSV file created at: {sample_csv_file_path}')

# 2. Function to run the Flask app on an alternative port to avoid 'Address already in use'
test_port = 5004  # Changed port to 5004 to avoid 'Address already in use'

def run_flask():
    # Set debug=False when running in a thread to avoid signal issues
    app.run(host='0.0.0.0', port=test_port, debug=False, use_reloader=False)

# 3. Start Flask in a background thread
flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

# 4. Display the UI directly in an iframe using Colab's built-in tool (no IP required)
print('\n--- TESTING GUIDE ---')
print('1. Use the interactive interface below to test the application. This method does NOT require an IP address or password.')
print(f'2. If the iframe does not load, you can try opening http://localhost:{test_port} manually, but this will require further tunneling setup if accessing from outside Colab.')
print('3. The application is now a single page where you enter all details and upload the recipient file.')
print(f'4. Use the generated \'{sample_excel_file_path}\' or \'{sample_csv_file_path}\' files to test the recipient list upload.')
print('5. Enter your SMTP credentials (e.g., Gmail SMTP with an App Password).')
print('----------------------')

output.serve_kernel_port_as_iframe(test_port)

# The previous localtunnel process is deprecated. Colab's iframe serving is prioritized for ease of use and no IP requirement.

Sample testing Excel file created at: sample_recipients.xlsx
Sample testing CSV file created at: sample_recipients.csv

--- TESTING GUIDE ---
1. Use the interactive interface below to test the application. This method does NOT require an IP address or password.
2. If the iframe does not load, you can try opening http://localhost:5004 manually, but this will require further tunneling setup if accessing from outside Colab.
3. The application is now a single page where you enter all details and upload the recipient file.
4. Use the generated 'sample_recipients.xlsx' or 'sample_recipients.csv' files to test the recipient list upload.
5. Enter your SMTP credentials (e.g., Gmail SMTP with an App Password).
----------------------


<IPython.core.display.Javascript object>

## Summary:

### Q&A

**What are the core features of the Bulk Email Sender application?**
The application is a Flask-based web tool that allows users to:
*   Configure SMTP server settings (host, port, credentials) via a web UI.
*   Upload recipient lists in Excel format (`.xlsx`, `.xls`).
*   Create personalized email templates using `{{name}}` placeholders for mail merging.
*   Send bulk emails securely using TLS encryption.
*   Receive a summary report of successful and failed deliveries.

**How does the application handle data processing and mail merging?**
The backend uses the `pandas` library to parse uploaded Excel files, specifically looking for "Email" and "Name" columns. During the sending process, it iterates through each row, replaces the `{{name}}` placeholder in the message body with the corresponding value from the "Name" column (defaulting to "Customer" if blank), and dispatches the email via `smtplib`.

---

### Data Analysis Key Findings

*   **Technology Stack:** The solution integrates **Flask** for the web framework, **Pandas** for data manipulation, and **Bootstrap 5** for a responsive frontend design.
*   **Robust Data Extraction:** The logic successfully handles missing data by defaulting null "Name" values to "Customer," preventing application crashes during the mail-merge process.
*   **Security Protocols:** The email dispatch logic utilizes `smtplib.SMTP` with `server.starttls()`, ensuring that credentials and email content are encrypted during transmission.
*   **Asynchronous Execution in Colab:** To run a web server within a notebook environment, the application was deployed using Python's `threading` module on port **5001**, bypassing standard blocking behavior.
*   **Error Resilience:** The sending loop is wrapped in individual try-except blocks, ensuring that a single invalid email address does not terminate the entire bulk job.

---

### Insights or Next Steps

*   **Next Step:** Implement an asynchronous task queue (like Celery or a simple background threading queue) for the sending process. Currently, the UI may hang or timeout while waiting for a large list of emails to finish sending, as the process is synchronous within the route.
*   **Insight:** For production use with Gmail, users must use "App Passwords" rather than their standard account password due to modern security requirements (OAuth2/MFA), which should be highlighted in the user documentation.
